# 📊 Parte 2 - Análise de Medalhas Olímpicas por Continente (1896-2024)

## 🎯 Objetivos

- 2.1 - Distribuição total de medalhas por continente
- 2.2 - Crescimento da representação ao longo do tempo
- 2.3 - Participação feminina por continente
- 2.4 - Modalidades mais fortes por continente
- 2.5 - Crescimento nas medalhas entre 1896 e 2024

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
import json
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (14,8)
plt.rcParams['font.size'] = 11
%matplotlib inline
print('✅ OK!')

In [ ]:
BASE_PATH=Path('../..')
RAW_PATH=BASE_PATH/'raw'
BRONZE_PATH=BASE_PATH/'bronze'/'parte2'
GOLD_PATH=BASE_PATH/'gold'/'parte2'
METADATA_PATH=BASE_PATH/'metadata'/'parte2'
OUTPUTS_PATH=BASE_PATH/'outputs'
for p in [BRONZE_PATH,GOLD_PATH,METADATA_PATH,(OUTPUTS_PATH/'figures'),(OUTPUTS_PATH/'tables')]:p.mkdir(parents=True,exist_ok=True)
print('✅ Paths!')

In [ ]:
import sys
sys.path.insert(0,str(BASE_PATH/'code'/'parte2'))
from noc_to_continent import get_continent,get_continents_summary
print('✅ Mapeamento!')
for c,n in sorted(get_continents_summary().items()):print(f'{c}:{n}')

## 📥 Carregamento e Integração

In [ ]:
print('CARREGANDO...')
df_hist_results=pd.read_csv(RAW_PATH/'Olympics_1896_2022'/'world_olympedia_olympics_athlete_event_result.csv')
df_hist_athletes=pd.read_csv(RAW_PATH/'Olympics_1896_2022'/'world_olympedia_olympics_athlete_bio.csv')
df_hist_games=pd.read_csv(RAW_PATH/'Olympics_1896_2022'/'world_olympedia_olympics_game.csv')
df_paris=pd.read_csv(RAW_PATH/'Olympics_Paris2024'/'medallists.csv')
print(f'✅ {len(df_hist_results):,} + {len(df_paris):,}')

In [ ]:
print('INTEGRANDO...')
df_h=df_hist_results[df_hist_results['medal'].notna()].copy()
df_h=df_h.merge(df_hist_games[['game_id','game_year','game_season']],on='game_id')
df_h=df_h.merge(df_hist_athletes[['athlete_id','athlete_sex']],on='athlete_id',how='left')
df_h['continent']=df_h['country_noc'].apply(get_continent)
df_p=df_paris.copy()
df_p['game_year']=2024
df_p['game_season']='Summer'
df_p['continent']=df_p['country'].apply(get_continent)
df_p=df_p.rename(columns={'country':'country_noc','medal_type':'medal','discipline':'event_discipline','gender':'athlete_sex'})
cols=['country_noc','medal','event_discipline','athlete_sex','game_year','game_season','continent']
df_medals=pd.concat([df_h[cols],df_p[cols]],ignore_index=True)
df_medals=df_medals[df_medals['continent']!='Desconhecido']
df_medals['medal']=df_medals['medal'].str.upper()
print(f'✅ {len(df_medals):,} medalhas')
df_medals.to_parquet(BRONZE_PATH/'medals_integrated.parquet',index=False)
print('💾 Bronze')

## 📊 2.1 - Distribuição de Medalhas por Continente

In [ ]:
print('='*80)
print('2.1 - DISTRIBUIÇÃO')
print('='*80)

# Total por continente
medals_cont=df_medals.groupby('continent')['medal'].count().sort_values(ascending=False)
print('\nTotal por continente:')
for c,n in medals_cont.items():print(f'{c:12}: {n:,}')

# Pizza
plt.figure(figsize=(10,10))
plt.pie(medals_cont.values,labels=medals_cont.index,autopct='%1.1f%%',startangle=90)
plt.title('Distribuição Total de Medalhas por Continente (1896-2024)',fontsize=14,fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUTS_PATH/'figures'/'parte2_pizza_continente.png',dpi=300,bbox_inches='tight')
print('\n💾 Gráfico: parte2_pizza_continente.png')
plt.show()

# Evolução temporal
medals_year=df_medals.groupby(['game_year','continent']).size().unstack(fill_value=0)

plt.figure(figsize=(16,8))
for cont in medals_year.columns:
    plt.plot(medals_year.index,medals_year[cont],marker='o',label=cont,linewidth=2)
plt.xlabel('Ano',fontsize=12)
plt.ylabel('Número de Medalhas',fontsize=12)
plt.title('Evolução de Medalhas por Continente (1896-2024)',fontsize=14,fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True,alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUTS_PATH/'figures'/'parte2_evolucao_continente.png',dpi=300,bbox_inches='tight')
print('💾 Gráfico: parte2_evolucao_continente.png')
plt.show()

# Salvar Gold
medals_cont.to_frame('total').to_parquet(GOLD_PATH/'medals_por_continente.parquet')
medals_year.to_parquet(GOLD_PATH/'medals_evolucao_temporal.parquet')
print('\n💾 Gold: medals_por_continente.parquet')
print('💾 Gold: medals_evolucao_temporal.parquet')

## 📈 2.2 - Crescimento da Representação

In [ ]:
print('='*80)
print('2.2 - CRESCIMENTO DA REPRESENTAÇÃO')
print('='*80)

# Países por ano
countries_year=df_medals.groupby(['game_year','continent'])['country_noc'].nunique().unstack(fill_value=0)
stats=countries_year.agg(['mean','std','min','max'])

print('\nEstatísticas (países participantes por continente):')
for cont in stats.columns:
    print(f'{cont:12}: média={stats.loc["mean",cont]:.1f}, std={stats.loc["std",cont]:.1f}, min={stats.loc["min",cont]:.0f}, max={stats.loc["max",cont]:.0f}')

# Gráfico
plt.figure(figsize=(16,8))
for cont in countries_year.columns:
    plt.plot(countries_year.index,countries_year[cont],marker='o',
             label=f'{cont} (média: {stats.loc["mean",cont]:.1f})',linewidth=2)
plt.xlabel('Ano',fontsize=12)
plt.ylabel('Número de Países',fontsize=12)
plt.title('Crescimento da Representação por Continente (1896-2024)',fontsize=14,fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True,alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUTS_PATH/'figures'/'parte2_crescimento_representacao.png',dpi=300,bbox_inches='tight')
print('\n💾 Gráfico: parte2_crescimento_representacao.png')
plt.show()

# Salvar
stats.to_parquet(GOLD_PATH/'crescimento_stats.parquet')
countries_year.to_parquet(GOLD_PATH/'crescimento_temporal.parquet')
print('💾 Gold: crescimento_stats.parquet')

## 👩 2.3 - Participação Feminina

In [ ]:
print('='*80)
print('2.3 - PARTICIPAÇÃO FEMININA')
print('='*80)

# Filtrar dados com sexo conhecido
df_gender=df_medals[df_medals['athlete_sex'].notna()].copy()

# Calcular % feminino por ano e continente
gender_counts=df_gender.groupby(['game_year','continent','athlete_sex']).size().unstack(fill_value=0)
total_by_year_cont=gender_counts.groupby(level=[0,1]).sum()

if 'F' in gender_counts.columns:
    female_counts=gender_counts['F'] if isinstance(gender_counts['F'],pd.Series) else gender_counts.xs('F',axis=1,level=1)
    female_pct=(female_counts/total_by_year_cont*100).unstack(fill_value=0)
    
    print('\n% Participação Feminina (últimas edições):')
    recent=female_pct.tail(5)
    print(recent.round(1))
    
    # Gráfico
    plt.figure(figsize=(16,8))
    for cont in female_pct.columns:
        plt.plot(female_pct.index,female_pct[cont],marker='o',label=cont,linewidth=2)
    plt.xlabel('Ano',fontsize=12)
    plt.ylabel('% de Participação Feminina',fontsize=12)
    plt.title('Evolução da Participação Feminina por Continente (1896-2024)',fontsize=14,fontweight='bold')
    plt.legend(fontsize=10)
    plt.grid(True,alpha=0.3)
    plt.ylim(0,100)
    plt.tight_layout()
    plt.savefig(OUTPUTS_PATH/'figures'/'parte2_participacao_feminina.png',dpi=300,bbox_inches='tight')
    print('\n💾 Gráfico: parte2_participacao_feminina.png')
    plt.show()
    
    # Salvar
    female_pct.to_parquet(GOLD_PATH/'participacao_feminina.parquet')
    print('💾 Gold: participacao_feminina.parquet')
else:
    print('⚠️ Dados de sexo não disponíveis')

## 🏅 2.4 - Modalidades Mais Fortes

In [ ]:
print('='*80)
print('2.4 - MODALIDADES MAIS FORTES')
print('='*80)

# Top 10 por continente
top_sports={}
for cont in df_medals['continent'].unique():
    df_c=df_medals[df_medals['continent']==cont]
    top10=df_c['event_discipline'].value_counts().head(10)
    top_sports[cont]=top10
    print(f'\n{cont}:')
    for i,(sport,count) in enumerate(top10.items(),1):
        print(f'  {i:2}. {sport:30} - {count:,} medalhas')

# Visualizar
fig,axes=plt.subplots(2,3,figsize=(20,12))
axes=axes.flatten()

for idx,(cont,sports) in enumerate(sorted(top_sports.items())):
    if idx<6:
        sports.plot(kind='barh',ax=axes[idx],color=sns.color_palette('Set2')[idx])
        axes[idx].set_title(f'Top 10 Modalidades - {cont}',fontsize=12,fontweight='bold')
        axes[idx].set_xlabel('Número de Medalhas',fontsize=10)
        axes[idx].set_ylabel('')
        axes[idx].grid(axis='x',alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUTS_PATH/'figures'/'parte2_modalidades_continente.png',dpi=300,bbox_inches='tight')
print('\n💾 Gráfico: parte2_modalidades_continente.png')
plt.show()

# Salvar
df_top=pd.DataFrame(top_sports)
df_top.to_parquet(GOLD_PATH/'modalidades_fortes.parquet')
print('💾 Gold: modalidades_fortes.parquet')

## 📊 2.5 - Crescimento nas Medalhas (1896-2024)

In [ ]:
print('='*80)
print('2.5 - CRESCIMENTO 1896-2024')
print('='*80)

# Comparar períodos
primerio=df_medals[df_medals['game_year']<=1920]
ultimo=df_medals[df_medals['game_year']>=2000]

crescimento=pd.DataFrame({
    '1896-1920':primerio.groupby('continent').size(),
    '2000-2024':ultimo.groupby('continent').size()
})
crescimento=crescimento.fillna(0)
crescimento['crescimento_%']=((crescimento['2000-2024']/crescimento['1896-1920'])-1)*100
crescimento['crescimento_%']=crescimento['crescimento_%'].replace([np.inf,-np.inf],np.nan)

print('\nComparação de períodos:')
print(crescimento.sort_values('crescimento_%',ascending=False).round(1))

# Gráfico
fig,ax=plt.subplots(figsize=(12,6))
crescimento[['1896-1920','2000-2024']].plot(kind='bar',ax=ax,color=['#8dd3c7','#fb8072'])
ax.set_title('Crescimento de Medalhas por Continente: 1896-1920 vs 2000-2024',fontsize=14,fontweight='bold')
ax.set_ylabel('Número de Medalhas',fontsize=12)
ax.set_xlabel('Continente',fontsize=12)
ax.set_xticklabels(crescimento.index,rotation=45,ha='right')
ax.legend(['1896-1920','2000-2024'])
ax.grid(axis='y',alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUTS_PATH/'figures'/'parte2_crescimento_1896_2024.png',dpi=300,bbox_inches='tight')
print('\n💾 Gráfico: parte2_crescimento_1896_2024.png')
plt.show()

# Salvar
crescimento.to_parquet(GOLD_PATH/'crescimento_periodos.parquet')
print('💾 Gold: crescimento_periodos.parquet')

## 💾 Gerando Metadados

In [ ]:
print('='*80)
print('GERANDO METADADOS')
print('='*80)

def save_metadata(filename,name,desc):
    meta={'nome':name,'descricao':desc,'periodo':'1896-2024','fonte':'Olympedia + Olympics.com','criado_em':datetime.now().isoformat()}
    with open(METADATA_PATH/f'{filename}_metadata.json','w') as f:json.dump(meta,f,indent=2)
    print(f'✅ {filename}_metadata.json')

save_metadata('medals_por_continente','Medalhas por Continente','Total de medalhas acumuladas por continente')
save_metadata('crescimento_stats','Estatísticas de Crescimento','Média, desvio padrão, min, max de países por continente')
save_metadata('participacao_feminina','Participação Feminina','Percentual de mulheres por continente ao longo do tempo')
save_metadata('modalidades_fortes','Modalidades Mais Fortes','Top 10 esportes que rendem mais medalhas por continente')
save_metadata('crescimento_periodos','Crescimento por Períodos','Comparação 1896-1920 vs 2000-2024')

print('\n✅ PARTE 2 COMPLETA!')

## ✅ Análise Concluída!

### Outputs Gerados:

**Bronze:**
- medals_integrated.parquet

**Gold:**
- medals_por_continente.parquet
- medals_evolucao_temporal.parquet
- crescimento_stats.parquet
- crescimento_temporal.parquet
- participacao_feminina.parquet
- modalidades_fortes.parquet
- crescimento_periodos.parquet

**Figuras:**
- parte2_pizza_continente.png
- parte2_evolucao_continente.png
- parte2_crescimento_representacao.png
- parte2_participacao_feminina.png
- parte2_modalidades_continente.png
- parte2_crescimento_1896_2024.png

**Metadados:**
- 5 arquivos JSON em metadata/parte2/